In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


In [3]:
import json
from pydantic import BaseModel, ValidationError
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam


class Car(BaseModel):
    brand: str
    price: float
    color: str


client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)


def extract_car(description: str) -> Car:
    messages: list[ChatCompletionMessageParam] = [
        {
            "role": "system",
            "content": (
                "你是一个信息抽取器。"
                "请从用户描述中抽取汽车信息，并且只返回 JSON。"
                "不要返回 Markdown，不要返回解释文字。"
                "JSON 必须包含以下字段：brand, price, color。"
                "brand 是汽车品牌，例如 宝马、奔驰、特斯拉。"
                "price 是数字类型，单位是人民币元。"
                "如果用户说 30万，price 返回 300000。"
                "color 是汽车颜色，例如 黑色、白色、红色。"
                "返回格式示例："
                '{"brand": "宝马", "price": 300000, "color": "黑色"}'
            ),
        },
        {
            "role": "user",
            "content": description,
        },
    ]

    completion = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=0,
    )

    content = completion.choices[0].message.content

    if content is None:
        raise ValueError("模型没有返回内容")

    try:
        data = json.loads(content)
    except json.JSONDecodeError as e:
        raise ValueError(f"模型返回的不是合法 JSON: {content}") from e

    try:
        car = Car.model_validate(data)
    except ValidationError as e:
        raise ValueError(f"模型返回的 JSON 不符合 Car 类型: {data}") from e

    return car

In [6]:
car = extract_car("今天天气不错，心情很好，我想买一辆绿色宝马，预算大概30万")
print(car)
print(type(car))
print(car.brand)
print(car.price)
print(car.color)

brand='宝马' price=300000.0 color='绿色'
<class '__main__.Car'>
宝马
300000.0
绿色
